<a href="https://colab.research.google.com/github/OlhaZahrebelna/certflow-rag-assistant/blob/main/src/ingestion/chunker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pyyaml

In [2]:
!git clone https://github.com/OlhaZahrebelna/certflow-rag-assistant.git

Cloning into 'certflow-rag-assistant'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 114 (delta 33), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 220.81 KiB | 2.87 MiB/s, done.
Resolving deltas: 100% (33/33), done.


In [3]:
%cd /content/certflow-rag-assistant

/content/certflow-rag-assistant


In [4]:
import re
import json
from pathlib import Path
import yaml

In [5]:
data_dir = Path("data/raw/source_markdown")

print(data_dir.exists())

True


In [6]:
files = list(data_dir.glob("*.md"))

for file in files:
    print(file.name)

print(f"\nTotal documents: {len(files)}")

06_address_certification_and_duplicate_prevention.md
09_frequently_asked_questions.md
03_end_to_end_certification_workflow.md
04_account_fields_and_validation_rules.md
10_policy_change_log.md
02_roles_and_responsibilities.md
08_quality_review_exceptions_and_escalations.md
05_source_hierarchy_and_evidence.md
01_account_certification_overview.md
07_request_types_and_change_management.md

Total documents: 10


In [7]:
def load_markdown_document(file_path: str) -> tuple[dict, str]:
    """
    Load metadata and content from a Markdown document.

    Expected format:

    ---
    document_id: ACD-KB-001
    title: Account Data Certification Overview
    ...
    ---

    # Account Data Certification Overview
    ...
    """

    path = Path(file_path)

    text = path.read_text(encoding="utf-8")

    # Split YAML front matter from Markdown content
    parts = text.split("---", 2)

    if len(parts) != 3:
        raise ValueError(
            f"File {file_path} does not contain valid YAML front matter."
        )

    metadata_text = parts[1]
    content = parts[2].strip()

    metadata = yaml.safe_load(metadata_text)

    return metadata, content

In [8]:
def split_into_sections(content: str) -> list[dict]:
    """
    Split Markdown document by level-2 headings (##).
    """

    pattern = r"(?m)^##\s+(.+)$"

    matches = list(re.finditer(pattern, content))

    sections = []

    for i, match in enumerate(matches):

        section_title = match.group(1).strip()

        start = match.end()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(content)

        section_content = content[start:end].strip()

        sections.append(
            {
                "section_title": section_title,
                "content": section_content,
            }
        )

    return sections

In [9]:
def create_chunks(
    metadata: dict,
    sections: list[dict]
) -> list[dict]:

    chunks = []

    for index, section in enumerate(sections):

        chunk_id = (
            f"{metadata['document_id']}-"
            f"chunk-{index + 1:03d}"
        )

        chunk = {
            "chunk_id": chunk_id,

            "content": section["content"],

            "metadata": {
                **metadata,

                "section": section["section_title"],

                "section_number": index + 1
            }
        }

        chunks.append(chunk)

    return chunks

In [10]:
def process_document(file_path: str) -> list[dict]:

    metadata, content = load_markdown_document(file_path)

    sections = split_into_sections(content)

    chunks = create_chunks(
        metadata=metadata,
        sections=sections
    )

    return chunks

In [11]:
all_chunks = []

for file_path in data_dir.glob("*.md"):
    chunks = process_document(file_path)

    all_chunks.extend(chunks)

    print(f"{file_path.name}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

06_address_certification_and_duplicate_prevention.md: 9 chunks
09_frequently_asked_questions.md: 5 chunks
03_end_to_end_certification_workflow.md: 10 chunks
04_account_fields_and_validation_rules.md: 9 chunks
10_policy_change_log.md: 4 chunks
02_roles_and_responsibilities.md: 8 chunks
08_quality_review_exceptions_and_escalations.md: 9 chunks
05_source_hierarchy_and_evidence.md: 8 chunks
01_account_certification_overview.md: 7 chunks
07_request_types_and_change_management.md: 7 chunks

Total chunks: 76


In [12]:
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "chunks.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        all_chunks,
        f,
        ensure_ascii=False,
        indent=2,
        default=str
    )

print(f"Saved {len(all_chunks)} chunks to {output_path}")

Saved 76 chunks to data/processed/chunks.json


In [13]:
print(output_path.exists())
print(output_path)

True
data/processed/chunks.json
